# 🌍 Transición Eléctrica Global de la Industria Automotriz (2015–2024)

**Datos:** dataset **sintético pero realista**, generado a partir de tendencias públicas conocidas de adopción de EVs, emisiones y especificaciones por marca (no se usa una API en vivo aquí para mantener el foco 100% en la capa de visualización).

Este dashboard combina **3 visualizaciones Plotly sincronizadas** controladas por los mismos widgets:

1. 🗺️ **Mapa mundial (choropleth)** — % de adopción de vehículos eléctricos por país.
2. 🧊 **Scatter 3D** — Relación Peso / Potencia / Precio, coloreado por emisiones de CO₂.
3. 🔥 **Heatmap + tendencia** — Evolución de emisiones de CO₂ por marca a través del tiempo.

Mover el **slider de año** o las selecciones de **marca/país** actualiza los tres gráficos *en el mismo instante* sin recrearlos desde cero (uso de `go.FigureWidget`).

> ⚠️ Nota de transparencia: los valores numéricos son simulados para fines de demostración de portafolio; los patrones (adopción creciente de EV, reducción de CO₂, liderazgo de ciertos países) están inspirados en tendencias reales de la industria, pero no deben tomarse como cifras oficiales.

In [13]:
# Importaciones y configuración
import numpy as np
import pandas as pd

import plotly.graph_objects as go
from plotly.subplots import make_subplots
import plotly.io as pio
import ipywidgets as widgets
from IPython.display import display

pio.renderers.default = "notebook"
np.random.seed(42)

print("Configuración lista.")

Configuración lista.


In [14]:
# Generación del dataset sintético (países x años) — adopción de EV

PAISES = {
    "China": "CHN", "Estados Unidos": "USA", "Alemania": "DEU", "Noruega": "NOR",
    "Japón": "JPN", "Corea del Sur": "KOR", "Reino Unido": "GBR", "Francia": "FRA",
    "India": "IND", "México": "MEX", "Brasil": "BRA", "Canadá": "CAN", "Italia": "ITA"
}
ANIOS = list(range(2015, 2025))

# Tasa de adopción "base" 2024 y velocidad de crecimiento por país (curva logística)
adopcion_2024 = {
    "China": 0.38, "Estados Unidos": 0.10, "Alemania": 0.22, "Noruega": 0.90,
    "Japón": 0.04, "Corea del Sur": 0.12, "Reino Unido": 0.20, "Francia": 0.18,
    "India": 0.03, "México": 0.02, "Brasil": 0.03, "Canadá": 0.11, "Italia": 0.09
}

filas_paises = []
for pais, iso3 in PAISES.items():
    techo = adopcion_2024[pais]
    velocidad = np.random.uniform(0.55, 0.75)
    for anio in ANIOS:
        t = (anio - 2015) / (2024 - 2015)
        valor = techo / (1 + np.exp(-10 * velocidad * (t - 0.65)))
        ruido = np.random.normal(0, 0.004)
        pct = max(0, valor + ruido) * 100
        filas_paises.append({"Pais": pais, "ISO3": iso3, "Anio": anio, "Adopcion_EV_pct": round(pct, 2)})

df_paises = pd.DataFrame(filas_paises)
df_paises.head()

,Pais,ISO3,Anio,Adopcion_EV_pct
0,China,CHN,2015,0.20
1,China,CHN,2016,1.39
2,China,CHN,2017,2.57
3,China,CHN,2018,5.02
4,China,CHN,2019,8.01


In [15]:
# Generación del dataset sintético (marcas x años) — specs y emisiones

MARCAS_INFO = {
    "Toyota":        {"segmento": "Masivo",    "co2_2015": 145, "co2_2024": 108, "potencia": 140, "peso": 1450, "precio": 28000},
    "Volkswagen":    {"segmento": "Masivo",    "co2_2015": 150, "co2_2024": 100, "potencia": 150, "peso": 1500, "precio": 30000},
    "Ford":          {"segmento": "Masivo",    "co2_2015": 160, "co2_2024": 118, "potencia": 220, "peso": 1750, "precio": 35000},
    "Honda":         {"segmento": "Masivo",    "co2_2015": 140, "co2_2024": 102, "potencia": 138, "peso": 1400, "precio": 27000},
    "Hyundai":       {"segmento": "Masivo",    "co2_2015": 142, "co2_2024":  95, "potencia": 145, "peso": 1420, "precio": 26000},
    "BMW":           {"segmento": "Lujo",      "co2_2015": 155, "co2_2024": 105, "potencia": 280, "peso": 1750, "precio": 55000},
    "Mercedes-Benz":  {"segmento": "Lujo",      "co2_2015": 158, "co2_2024": 108, "potencia": 300, "peso": 1800, "precio": 58000},
    "Tesla":         {"segmento": "Eléctrico", "co2_2015":   0, "co2_2024":   0, "potencia": 450, "peso": 1900, "precio": 47000},
    "BYD":           {"segmento": "Eléctrico", "co2_2015":  60, "co2_2024":   0, "potencia": 380, "peso": 1850, "precio": 32000},
    "Porsche":       {"segmento": "Deportivo", "co2_2015": 210, "co2_2024": 140, "potencia": 420, "peso": 1650, "precio": 95000},
}

filas_marcas = []
for marca, info in MARCAS_INFO.items():
    for anio in ANIOS:
        t = (anio - 2015) / (2024 - 2015)
        co2 = info["co2_2015"] + (info["co2_2024"] - info["co2_2015"]) * t + np.random.normal(0, 2.5)
        co2 = max(0, co2)
        potencia = info["potencia"] * (1 + 0.15 * t) + np.random.normal(0, 4)
        peso = info["peso"] * (1 + 0.05 * t) + np.random.normal(0, 15)
        precio = info["precio"] * (1 + 0.25 * t) + np.random.normal(0, 500)
        ventas = np.random.uniform(0.3, 3.5) * (1.4 if info["segmento"] == "Masivo" else 1.0)
        filas_marcas.append({
            "Marca": marca, "Segmento": info["segmento"], "Anio": anio,
            "CO2_g_km": round(co2, 1), "Potencia_hp": round(potencia, 0),
            "Peso_kg": round(peso, 0), "Precio_usd": round(precio, 0),
            "Ventas_millones": round(ventas, 2)
        })

df_marcas = pd.DataFrame(filas_marcas)
df_marcas.head()

,Marca,Segmento,Anio,CO2_g_km,Potencia_hp,Peso_kg,Precio_usd,Ventas_millones
0,Toyota,Masivo,2015,144.0,136.0,1445.0,28278.0,3.38
1,Toyota,Masivo,2016,141.5,148.0,1434.0,28870.0,0.84
2,Toyota,Masivo,2017,133.2,149.0,1486.0,29122.0,4.44
3,Toyota,Masivo,2018,133.3,148.0,1464.0,30449.0,1.14
4,Toyota,Masivo,2019,129.1,153.0,1485.0,30503.0,3.52


In [16]:
# Widgets de control (compartidos por los 3 gráficos)

lista_marcas = sorted(df_marcas["Marca"].unique())
lista_paises = sorted(df_paises["Pais"].unique())

w_anio = widgets.IntSlider(value=2024, min=min(ANIOS), max=max(ANIOS), step=1,
                            description="Año:", continuous_update=False)
w_play = widgets.Play(value=2024, min=min(ANIOS), max=max(ANIOS), step=1, interval=900, description="▶")
widgets.jslink((w_play, "value"), (w_anio, "value"))

w_marcas = widgets.SelectMultiple(options=lista_marcas, value=tuple(lista_marcas[:5]),
                                   description="Marcas:", rows=6)
w_paises = widgets.SelectMultiple(options=lista_paises, value=tuple(lista_paises),
                                   description="Países:", rows=6)

controles_db3 = widgets.VBox([
    widgets.HBox([w_play, w_anio]),
    widgets.HBox([w_marcas, w_paises])
])
display(controles_db3)

In [17]:
# Construcción inicial de las 3 figuras como FigureWidget (para actualizarlas in-place)

anio_inicial = w_anio.value
paises_ini = list(w_paises.value)
marcas_ini = list(w_marcas.value)

# --- Figura A: Choropleth de adopción de EV ---
df_mapa_ini = df_paises[(df_paises["Anio"] == anio_inicial) & (df_paises["Pais"].isin(paises_ini))]
fig_mapa = go.FigureWidget(
    data=[go.Choropleth(
        locations=df_mapa_ini["ISO3"], z=df_mapa_ini["Adopcion_EV_pct"],
        text=df_mapa_ini["Pais"], colorscale="Teal", zmin=0, zmax=90,
        colorbar_title="% Adopción EV", marker_line_color="white"
    )],
    layout=go.Layout(
        title=f"Adopción de Vehículos Eléctricos por País — {anio_inicial}",
        geo=dict(showframe=False, showcoastlines=True, projection_type="natural earth"),
        height=430, margin=dict(l=10, r=10, t=50, b=10)
    )
)

# --- Figura B: Scatter 3D Peso / Potencia / Precio, color = CO2 ---
df_3d_ini = df_marcas[(df_marcas["Anio"] == anio_inicial) & (df_marcas["Marca"].isin(marcas_ini))]
fig_3d = go.FigureWidget(
    data=[go.Scatter3d(
        x=df_3d_ini["Peso_kg"], y=df_3d_ini["Potencia_hp"], z=df_3d_ini["Precio_usd"],
        mode="markers+text", text=df_3d_ini["Marca"], textposition="top center",
        marker=dict(
            size=df_3d_ini["Ventas_millones"] * 6 + 4,
            color=df_3d_ini["CO2_g_km"], colorscale="RdYlGn_r", colorbar=dict(title="CO₂ g/km"),
            line=dict(width=0.5, color="white")
        )
    )],
    layout=go.Layout(
        title=f"Peso vs. Potencia vs. Precio por Marca — {anio_inicial}",
        scene=dict(xaxis_title="Peso (kg)", yaxis_title="Potencia (hp)", zaxis_title="Precio (USD)"),
        height=480, margin=dict(l=0, r=0, t=50, b=0)
    )
)

# --- Figura C: Heatmap (Marca x Año, CO2) + línea de tendencia sincronizada ---
pivot_co2 = df_marcas.pivot(index="Marca", columns="Anio", values="CO2_g_km")
fig_heat_line = make_subplots(
    rows=2, cols=1, shared_xaxes=False, row_heights=[0.55, 0.45],
    subplot_titles=("Mapa de calor: CO₂ (g/km) por Marca y Año", "Tendencia de CO₂ — marcas seleccionadas")
)
fig_heat_line.add_trace(go.Heatmap(
    z=pivot_co2.values, x=pivot_co2.columns, y=pivot_co2.index,
    colorscale="RdYlGn_r", colorbar=dict(title="CO₂", y=0.78, len=0.5)
), row=1, col=1)

for marca in marcas_ini:
    serie = df_marcas[df_marcas["Marca"] == marca]
    fig_heat_line.add_trace(go.Scatter(
        x=serie["Anio"], y=serie["CO2_g_km"], mode="lines+markers", name=marca
    ), row=2, col=1)

fig_heat_line.add_vline(x=anio_inicial, line_dash="dash", line_color="gray", row=2, col=1)
fig_heat_line.update_layout(height=650, margin=dict(l=10, r=10, t=60, b=10),
                             title_text="Emisiones de CO₂ por Marca a través del Tiempo")
fig_heat_line = go.FigureWidget(fig_heat_line)

display(fig_mapa)
display(fig_3d)
display(fig_heat_line)

FigureWidget({
    'data': [{'colorbar': {'title': {'text': '% Adopción EV'}},
              'colorscale': [[0.0, 'rgb(209, 238, 234)'], [0.16666666666666666,
                             'rgb(168, 219, 217)'], [0.3333333333333333, 'rgb(133,
                             196, 201)'], [0.5, 'rgb(104, 171, 184)'],
                             [0.6666666666666666, 'rgb(79, 144, 166)'],
                             [0.8333333333333334, 'rgb(59, 115, 143)'], [1.0,
                             'rgb(42, 86, 116)']],
              'locations': array(['CHN', 'USA', 'DEU', 'NOR', 'JPN', 'KOR', 'GBR', 'FRA', 'IND', 'MEX',
                                  'BRA', 'CAN', 'ITA'], dtype=object),
              'marker': {'line': {'color': 'white'}},
              'text': array(['China', 'Estados Unidos', 'Alemania', 'Noruega', 'Japón',
                             'Corea del Sur', 'Reino Unido', 'Francia', 'India', 'México', 'Brasil',
                             'Canadá', 'Italia'], dtype=object),
   

FigureWidget({
    'data': [{'marker': {'color': {'bdata': 'AAAAAACgXUCamZmZmTlaQM3MzMzMTFhAZmZmZmamWkAAAAAAAAAIQA==', 'dtype': 'f8'},
                         'colorbar': {'title': {'text': 'CO₂ g/km'}},
                         'colorscale': [[0.0, 'rgb(0,104,55)'], [0.1,
                                        'rgb(26,152,80)'], [0.2,
                                        'rgb(102,189,99)'], [0.3,
                                        'rgb(166,217,106)'], [0.4,
                                        'rgb(217,239,139)'], [0.5,
                                        'rgb(255,255,191)'], [0.6,
                                        'rgb(254,224,139)'], [0.7,
                                        'rgb(253,174,97)'], [0.8,
                                        'rgb(244,109,67)'], [0.9,
                                        'rgb(215,48,39)'], [1.0, 'rgb(165,0,38)']],
                         'line': {'color': 'white', 'width': 0.5},
                         'size': {'bdata': 

FigureWidget({
    'data': [{'colorbar': {'len': 0.5, 'title': {'text': 'CO₂'}, 'y': 0.78},
              'colorscale': [[0.0, 'rgb(0,104,55)'], [0.1, 'rgb(26,152,80)'],
                             [0.2, 'rgb(102,189,99)'], [0.3, 'rgb(166,217,106)'],
                             [0.4, 'rgb(217,239,139)'], [0.5, 'rgb(255,255,191)'],
                             [0.6, 'rgb(254,224,139)'], [0.7, 'rgb(253,174,97)'],
                             [0.8, 'rgb(244,109,67)'], [0.9, 'rgb(215,48,39)'],
                             [1.0, 'rgb(165,0,38)']],
              'type': 'heatmap',
              'uid': 'f88281a9-bba4-4638-a60f-fe78aae2a4b5',
              'x': {'bdata': '3wfgB+EH4gfjB+QH5QfmB+cH6Ac=', 'dtype': 'i2'},
              'xaxis': 'x',
              'y': array(['BMW', 'BYD', 'Ford', 'Honda', 'Hyundai', 'Mercedes-Benz', 'Porsche',
                          'Tesla', 'Toyota', 'Volkswagen'], dtype=object),
              'yaxis': 'y',
              'z': {'bdata': ('MzMzMzNTY0CamZmZmcli

In [18]:
# Función de sincronización — un solo callback actualiza las 3 figuras a la vez

def actualizar_dashboard(change=None):
    anio_sel = w_anio.value
    paises_sel = list(w_paises.value) if w_paises.value else lista_paises
    marcas_sel = list(w_marcas.value) if w_marcas.value else lista_marcas

    # --- Actualiza Figura A (mapa) ---
    df_mapa = df_paises[(df_paises["Anio"] == anio_sel) & (df_paises["Pais"].isin(paises_sel))]
    with fig_mapa.batch_update():
        fig_mapa.data[0].locations = df_mapa["ISO3"]
        fig_mapa.data[0].z = df_mapa["Adopcion_EV_pct"]
        fig_mapa.data[0].text = df_mapa["Pais"]
        fig_mapa.layout.title.text = f"Adopción de Vehículos Eléctricos por País — {anio_sel}"

    # --- Actualiza Figura B (3D) ---
    df_3d = df_marcas[(df_marcas["Anio"] == anio_sel) & (df_marcas["Marca"].isin(marcas_sel))]
    with fig_3d.batch_update():
        fig_3d.data[0].x = df_3d["Peso_kg"]
        fig_3d.data[0].y = df_3d["Potencia_hp"]
        fig_3d.data[0].z = df_3d["Precio_usd"]
        fig_3d.data[0].text = df_3d["Marca"]
        fig_3d.data[0].marker.size = df_3d["Ventas_millones"] * 6 + 4
        fig_3d.data[0].marker.color = df_3d["CO2_g_km"]
        fig_3d.layout.title.text = f"Peso vs. Potencia vs. Precio por Marca — {anio_sel}"

    # --- Actualiza Figura C (heatmap se queda fijo; las líneas y la marca de año se reconstruyen) ---
    with fig_heat_line.batch_update():
        # Elimina trazas de línea previas (todo excepto la traza 0, que es el heatmap)
        fig_heat_line.data = (fig_heat_line.data[0],)
        for marca in marcas_sel:
            serie = df_marcas[df_marcas["Marca"] == marca]
            fig_heat_line.add_trace(go.Scatter(
                x=serie["Anio"], y=serie["CO2_g_km"], mode="lines+markers", name=marca
            ), row=2, col=1)
        # Redibuja la línea vertical del año seleccionado
        fig_heat_line.layout.shapes = []
        fig_heat_line.add_vline(x=anio_sel, line_dash="dash", line_color="gray", row=2, col=1)

w_anio.observe(actualizar_dashboard, names="value")
w_marcas.observe(actualizar_dashboard, names="value")
w_paises.observe(actualizar_dashboard, names="value")

print("✅ Sincronización activa: mueve el slider de año (▶ para animar) o cambia marcas/países arriba.")

✅ Sincronización activa: mueve el slider de año (▶ para animar) o cambia marcas/países arriba.


---
### 💡 Notas técnicas
- Las 3 figuras usan `go.FigureWidget`, lo que permite actualizar `data` y `layout` **in-place** (`batch_update()`) en lugar de volver a renderizar el gráfico completo — esto es lo que da la sensación de "dashboard sincronizado" en vez de tres gráficos independientes.
- El widget `Play` está enlazado (`jslink`) al slider de año, permitiendo animar la transición 2015→2024 con un clic.
- El heatmap (fondo temporal completo) se mantiene fijo mientras que las líneas de tendencia y el marcador de año se reconstruyen dinámicamente según las marcas seleccionadas.
- Todos los datos son sintéticos (ver nota de transparencia al inicio); para producción se recomendaría sustituir por datos reales de fuentes como IEA Global EV Outlook o EPA.